In [ ]:
import torch
import sys
import os
sys.path.append(os.path.abspath(".."))
from src.datasets.data_loader import get_dataloaders
from src.models.fmnist_cnn import fmnistCNN
from src.models.cifar_cnn import cifarCNN

def test_pipeline(name, model_class):
    print(f"\n--- Testing {name} ---")
    train, val, test = get_dataloaders(name)
    
    # Get one batch
    images, labels = next(iter(train))
    print(f"Batch Shape: {images.shape}") # Should be [64, 1, 28, 28] or [64, 3, 32, 32]
    
    # Pass through model
    model = model_class()
    output = model(images)
    print(f"Model Output Shape: {output.shape}") # Should be [64, 10]

if __name__ == "__main__":
    test_pipeline("fashion_mnist", fmnistCNN)
    test_pipeline("cifar10", cifarCNN)



--- Testing fashion_mnist ---


100.0%
100.0%
100.0%
100.0%


Batch Shape: torch.Size([64, 1, 28, 28])
Model Output Shape: torch.Size([64, 10])

--- Testing cifar10 ---


100.0%
c:\Users\user\OneDrive\Desktop\University\Courses\CMPS\CMPS 299 (Final Year Project)\Comparative-Optimization-Project\venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Batch Shape: torch.Size([64, 3, 32, 32])
Model Output Shape: torch.Size([64, 10])


In [ ]:
import torch
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.datasets.data_loader import get_dataloaders
from src.models.fmnist_cnn import fmnistCNN
from src.models.cifar_cnn import cifarCNN

def run_comprehensive_test(name, model_class, expected_channels, expected_size):
    print(f"\n{'='*10} TESTING {name.upper()} {'='*10}")
    
    # 1. Load Data
    try:
        train_loader, val_loader, test_loader = get_dataloaders(name, batch_size=64)
        print("Loaders created successfully.")
    except Exception as e:
        print(f"Failed: {e}")
        return

    # 2. Check Split and Reproducibility
    # We load a second time to ensure the validation set is identical (Seed Check)
    _, val_loader_check, _ = get_dataloaders(name, batch_size=64)
    
    images1, labels1 = next(iter(val_loader))
    images2, labels2 = next(iter(val_loader_check))
    
    if torch.equal(labels1, labels2):
        print(f"Reproducibility confirmed (Val sets match).")
        print(f"   Sizes: Train={len(train_loader.dataset)}, Val={len(val_loader.dataset)}, Test={len(test_loader.dataset)}")
    else:
        print(f"Warning: Validation sets are NOT reproducible. Check your random_split seed!")

    # 3. Batch Shape & Data Integrity
    images, labels = next(iter(train_loader))
    print(f"Data Integrity")
    print(f"   - Image Shape: {images.shape} (Expected: [64, {expected_channels}, {expected_size}, {expected_size}])")
    print(f"   - Label Shape: {labels.shape} (Expected: [64])")
    print(f"   - Dtypes: Images={images.dtype}, Labels={labels.dtype}")
    print(f"   - Stats: Mean={images.mean():.3f}, Std={images.std():.3f}")
    print(f"   - Shuffle Check (First 5 labels): {labels[:5].tolist()}")

    # 4. Model Forward Pass
    try:
        model = model_class()
        output = model(images)
        print(f"Forward Pass successful. Output shape: {output.shape}")
        
        if output.shape == (64, 10):
            print("Conclusion: Dataset and Model are perfectly aligned.")
    except Exception as e:
        print(f"Model Mismatch: {e}")

if __name__ == "__main__":
    # Test Fashion-MNIST (Grayscale, 28x28)
    run_comprehensive_test("fashion_mnist", fmnistCNN, 1, 28)
    
    # Test CIFAR-10 (RGB, 32x32)
    run_comprehensive_test("cifar10", cifarCNN, 3, 32)


========== TESTING FASHION_MNIST ==========
Loaders created successfully.
Reproducibility confirmed (Val sets match).
   Sizes: Train=54000, Val=6000, Test=10000
Data Integrity
   - Image Shape: torch.Size([64, 1, 28, 28]) (Expected: [64, 1, 28, 28])
   - Label Shape: torch.Size([64]) (Expected: [64])
   - Dtypes: Images=torch.float32, Labels=torch.int64
   - Stats: Mean=-0.410, Std=0.717
   - Shuffle Check (First 5 labels): [6, 5, 3, 6, 1]
Forward Pass successful. Output shape: torch.Size([64, 10])
Conclusion: Dataset and Model are perfectly aligned.

========== TESTING CIFAR10 ==========
Loaders created successfully.
Reproducibility confirmed (Val sets match).
   Sizes: Train=45000, Val=5000, Test=10000
Data Integrity
   - Image Shape: torch.Size([64, 3, 32, 32]) (Expected: [64, 3, 32, 32])
   - Label Shape: torch.Size([64]) (Expected: [64])
   - Dtypes: Images=torch.float32, Labels=torch.int64
   - Stats: Mean=-0.077, Std=0.496
   - Shuffle Check (First 5 labels): [5, 5, 0, 1, 1]
F

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import sys
import os

sys.path.append(os.path.abspath(".."))


from src.datasets.data_loader import get_dataloaders
from src.models.fmnist_cnn import fmnistCNN        
from src.training.train import train_model            
from src.training.metrics import get_device

def run_test():
    device = get_device()
    print(f"Testing pipeline on: {device}")

    train_loader, val_loader, _ = get_dataloaders("fashion_mnist")

    model = fmnistCNN().to(device)

    lossfn = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    print("Starting mini-training...")
    try:
        trained_model, history = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            lossfn=lossfn,
            optimizer=optimizer,
            device=device,
            num_epochs=2  # Just 2 epochs to verify it works
        )
        print("\nTEST PASSED!")
        print(f"Final Val Accuracy: {history['val_acc'][-1]:.2f}%")
        
    except Exception as e:
        print(f"\nTEST FAILED!")
        print(f"Error: {e}")

if __name__ == "__main__":
    run_test()

Testing pipeline on: cpu
Starting mini-training...
Epoch [1/2] -> Train: 0.4453 (83.75%) | Val: 0.3222 (88.52%)
Epoch [2/2] -> Train: 0.2811 (89.72%) | Val: 0.2683 (89.93%)

TEST PASSED!
Final Val Accuracy: 89.93%


In [8]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.11.0+cpu
CUDA available: False
